In [1]:
import pandas as pd
import pyreadstat

FILE = r"C:/Users/Lenovo/London_Sport_2/data/raw/UKDA-8223-spss/spss/spss28/active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav"   # <-- your real path, raw string

# Full read once, so you can search the dictionary. Slow, but you only do it here.
df_full, meta = pyreadstat.read_sav(FILE)
print(df_full.shape)   # expect ~ (180000, 3165)

(198911, 3165)


In [2]:
# Two search tools for finding the variable you need among thousands.
# find_vars      -> searches the human-readable LABELS (the question text)
# find_var_names -> searches the variable NAMES themselves
# Both are needed because some variables (e.g. wt_final, LA) are only findable
# by name, while others are only obvious from their label.
def find_vars(meta, *keywords):           # search the LABELS
    return {v: l for v, l in meta.column_names_to_labels.items()
            if l and any(k.lower() in l.lower() for k in keywords)}

def find_var_names(meta, *keywords):      # search the NAMES
    return [v for v in meta.column_names_to_labels
            if any(k.lower() in v.lower() for k in keywords)]

In [3]:
# The ONLY thing that changes between survey waves. Each wave gets one entry
# mapping our standard concept names (weight, la_code, ...) to that wave's
# actual variable names. The code below never changes — only this dict does.
# When you add 2016_17, 2017_18, etc., look up their variable names with the
# find_vars helpers and add a new block here.
WAVE_CONFIG = {
    "2015_16": {
        "file": FILE,
        "weight": "wt_final",                       # main full-sample survey weight
        "la_code": "LA",                            # local authority (numeric; labels embed E09 GSS code)
        "activity_band": "MEMS7GR_SPORTCOUNT_A01",   # 0=Inactive 1=Insufficiently Active 2=Active; matches SE ~61%
        "age": "Age5",                              # age in 5 bands
        "gender": "Gend3",
        "ethnicity": "Eth7",                        # ethnicity in 7 groups
        "disability": "Disab3",                     # whether limiting disability
    },
}

In [4]:
# Keep only respondents in the 33 Greater London local authorities.
# LA is stored as numbers (1.0, 2.0, ...) whose value-LABELS embed the ONS GSS
# code, e.g. 9.0 -> 'E09000002 Barking and Dagenham'. All London codes start
# 'E09', so we find every numeric code whose label begins 'E09' and keep those
# rows. Keying off the GSS prefix (not borough names) avoids spelling mismatches.
def london_filter(df, meta, cfg):
    la = cfg["la_code"]
    labels = meta.variable_value_labels.get(la, {})   # {code: 'E09... Borough'}
    london_codes = [code for code, label in labels.items()
                    if str(label).startswith("E09")]  # the 33 London LAs
    if not london_codes:
        raise ValueError(f"No London codes found in value labels for {la!r}")
    return df[df[la].isin(london_codes)].copy()

In [5]:
# Turn raw survey columns into analysis-ready ones:
#  1) Replace the survey's missing-data codes (-90 to -99, e.g. "prefer not to
#     say", "not applicable") with real NaN so they don't get counted as data.
#     NOTE: we deliberately do NOT strip -1, which is a genuine value elsewhere.
#  2) Decode the coded columns we care about into readable labels using the
#     metadata dictionaries (e.g. 2 -> "Active", 1 -> "Female").
#  3) Force the weight to numeric, and copy the age band across.
# Uses cfg.get(key) so it silently skips any concept missing from the config —
# that's what lets the national sanity-check cells pass a trimmed-down config.
def clean_wave(df, meta, cfg, missing_codes=tuple(range(-99, -89))):  # -99..-90
    df = df.copy()
    df[df.isin(list(missing_codes))] = pd.NA          # missing codes -> NaN

    for key in ["activity_band", "gender", "ethnicity", "disability"]:
        var = cfg.get(key)
        if var and var in meta.variable_value_labels:
            df[key] = df[var].map(meta.variable_value_labels[var])  # decode to labels
        elif var:
            df[key] = df[var]

    df["weight"] = pd.to_numeric(df[cfg["weight"]], errors="coerce")
    if cfg.get("age"):
        df["age"] = df[cfg["age"]]   # Age5 is already a band — keep as-is
    return df

In [6]:
# Every population figure must use the survey weight, never a raw row count.
# A weighted share = (sum of weights where the condition is true)
#                    / (total weight in the group).
# If group_cols is given, it returns one row per group plus 'n', the unweighted
# sample size — use n to spot groups too small to trust (flag anything under ~30-50).
def weighted_share(df, group_cols, target_col, target_value):
    d = df.dropna(subset=["weight", target_col]).copy()
    d["_num"] = (d[target_col] == target_value).astype(float) * d["weight"]
    if group_cols:
        g = d.groupby(group_cols, dropna=False)
        out = (g["_num"].sum() / g["weight"].sum()).rename("weighted_share").reset_index()
        out["n"] = g.size().values          # unweighted base
        return out
    return d["_num"].sum() / d["weight"].sum()   # single overall figure

In [7]:
# One function that turns a raw wave into a clean, London-only DataFrame:
#   read only the columns we need -> filter to London -> clean/decode -> tag wave.
# usecols keeps memory down by returning only the configured variables.
# Returns the cleaned data plus its metadata. To process another wave later,
# just call process_wave("2016_17", WAVE_CONFIG) once that wave is in the config.
def process_wave(wave_key, config):
    cfg = config[wave_key]
    wanted = [cfg[k] for k in ["weight","la_code","activity_band","age",
                               "gender","ethnicity","disability"] if cfg.get(k)]
    df, meta = pyreadstat.read_sav(cfg["file"], usecols=wanted)
    df = london_filter(df, meta, cfg)
    df = clean_wave(df, meta, cfg)
    df["wave"] = wave_key      # so waves stay identifiable once stacked together
    return df, meta

In [8]:
%%time

# Execute the pipeline for 2015-16 and save the clean London data to parquet.
# parquet is a fast, compact format — later analysis can reload this instantly
# instead of reprocessing from the raw .sav. %%time prints how long it took.

london_2015_16, meta_2015_16 = process_wave("2015_16", WAVE_CONFIG)
london_2015_16.to_parquet("london_2015_16_clean.parquet")
print(london_2015_16.shape)   # London rows x columns

(19887, 14)
CPU times: total: 15.1 s
Wall time: 15.7 s


In [9]:
# Validation step: does our decoding + weighting reproduce Sport England's
# published national figure (~61% active for 2015-16)? We clean ONLY the
# activity band + weight from the full national sample (fast, 2 columns) using
# a trimmed config so clean_wave doesn't look for demographic columns we left out.
base = WAVE_CONFIG["2015_16"]
cfg = {"activity_band": base["activity_band"], "weight": base["weight"]}
nat_small = df_full[[cfg["activity_band"], cfg["weight"]]].copy()
nat = clean_wave(nat_small, meta, cfg)
print(weighted_share(nat, [], "activity_band", "Active"))

0.6207443376518277


In [10]:
# Active Lives has several "activity level" variables (different official
# definitions). We don't know which one matches the published headline, so we
# compute the national weighted % Active for each candidate. Whichever lands
# nearest ~0.61 is the definition Sport England reports — set that as
# 'activity_band' in WAVE_CONFIG. (MEMS7GR_ALL is coded as INactivity and may
# label its categories differently — check its value labels if it's the match.)
base = WAVE_CONFIG["2015_16"]
for v in ["MEMS7GR_SPORTFUND_A02", "MEMS7GR_SPORTCOUNT_A01",
          "MEMS7GR_SPORTPRE2016_A03", "MEMS7GR_ALL"]:
    cfg = {"activity_band": v, "weight": base["weight"]}
    small = df_full[[v, base["weight"]]].copy()
    n = clean_wave(small, meta, cfg)
    print(v, round(weighted_share(n, [], "activity_band", "Active"), 3))

MEMS7GR_SPORTFUND_A02 0.546
MEMS7GR_SPORTCOUNT_A01 0.621
MEMS7GR_SPORTPRE2016_A03 0.402
MEMS7GR_ALL 0.654


In [11]:
# The actual outputs for the project: London's overall weighted % Active, then
# broken down by ethnicity (uncomment the others for age and gender). Each
# breakdown row carries 'n' so you can see which categories have enough sample
# to be reliable. The NaN ethnicity row = respondents with no ethnicity recorded;
# check london_2015_16["ethnicity"].isna().mean() to see how big that group is.
print("Overall London % Active:",
      round(weighted_share(london_2015_16, [], "activity_band", "Active"), 3))

weighted_share(london_2015_16, ["ethnicity"], "activity_band", "Active")
# weighted_share(london_2015_16, ["age"], "activity_band", "Active")
# weighted_share(london_2015_16, ["gender"], "activity_band", "Active")

Overall London % Active: 0.635


,ethnicity,weighted_share,n
0,Black,0.576792,1297
1,Chinese,0.610678,319
2,Mixed,0.734020,522
3,Other ethnic group,0.556157,529
4,South Asian,0.547168,2365
5,White British,0.670972,10782
6,White Other,0.688909,2855
7,NaN,0.593150,1218


In [13]:
# Reusable data-quality audit for any processed wave. Runs the five checks:
#   1) missingness per analysis column   2) weight validity (positive, non-missing)
#   3) activity band contains only real categories (no leaked survey codes)
#   4) demographic labels decoded to readable text
#   5) all 33 London boroughs present with plausible sizes
# Prints a readable report and returns a dict of the headline numbers so you can
# compare waves at a glance. Diagnostic only — it never modifies the data.
def audit_wave(df, meta, wave_key="", la_var="LA"):
    print(f"===== AUDIT: {wave_key} =====")
    n = len(df)
    print(f"Rows: {n:,}")

    # 1) Missingness per column
    cols = [c for c in ["activity_band","ethnicity","gender","age","disability","weight"]
            if c in df.columns]
    miss = df[cols].isna().mean().round(3).sort_values(ascending=False)
    print("\n[1] Proportion missing per column:")
    print(miss.to_string())

    # 2) Weight validity
    w = df["weight"]
    n_bad = int((w <= 0).sum()); n_naw = int(w.isna().sum())
    print("\n[2] Weight: min={:.4f} max={:.2f} | non-positive={} missing={}".format(
        w.min(), w.max(), n_bad, n_naw))
    if n_bad or n_naw:
        print("    ^ WARNING: some rows can't be validly weighted.")

    # 3) Activity band categories (should be only the 3 real labels + NaN)
    print("\n[3] activity_band categories:")
    print(df["activity_band"].value_counts(dropna=False).to_string())

    # 4) Decoded demographic labels
    print("\n[4] Decoded label samples:")
    for c in ["ethnicity","gender","disability"]:
        if c in df.columns:
            print(f"    {c}: {list(df[c].dropna().unique())[:8]}")

    # 5) London borough coverage
    labels = meta.variable_value_labels.get(la_var, {})
    names = df[la_var].map(labels)
    n_la = names.nunique()
    print(f"\n[5] London LAs present: {n_la} (expect 33)")
    if n_la != 33:
        print("    ^ WARNING: borough count != 33 — check the filter or sample.")
    smallest = names.value_counts().tail(3)
    print("    Smallest boroughs by n:")
    print(smallest.to_string())

    print("=" * (14 + len(str(wave_key))), "\n")
    return {"wave": wave_key, "rows": n,
            "miss_activity": float(miss.get("activity_band", 0)),
            "miss_weight": float(miss.get("weight", 0)),
            "miss_ethnicity": float(miss.get("ethnicity", 0)),
            "bad_weights": n_bad + n_naw,
            "n_londons_las": n_la}

In [14]:
report_2015_16 = audit_wave(london_2015_16, meta_2015_16, wave_key="2015_16")

===== AUDIT: 2015_16 =====
Rows: 19,887

[1] Proportion missing per column:
disability       0.065
ethnicity        0.061
age              0.011
gender           0.002
activity_band    0.000
weight           0.000

[2] Weight: min=0.0183 max=37.01 | non-positive=0 missing=0

[3] activity_band categories:
activity_band
Active                   12687
Inactive                  4724
Insufficiently Active     2476

[4] Decoded label samples:
    ethnicity: ['White British', 'White Other', 'South Asian', 'Other ethnic group', 'Black', 'Mixed', 'Chinese']
    gender: ['Female', 'Male']
    disability: ['No disability', 'Non-limiting disability', 'Limiting disability']

[5] London LAs present: 33 (expect 33)
    Smallest boroughs by n:
LA
E09000017 Hillingdon        486
E09000025 Newham            486
E09000001 City of London    267



## Wave 1 (2015–16) — Findings & Data Quality

### What this notebook does
Takes the raw Active Lives 2015–16 SPSS file (198,911 respondents, 3,165 variables),
filters to the 33 Greater London local authorities, decodes and cleans the variables
of interest, audits data quality, and produces **survey-weighted** activity figures
overall and by ethnicity. The logic is wave-agnostic: only the `WAVE_CONFIG` entry
changes when adding later waves.

### Key decisions
- **Activity measure:** `MEMS7GR_SPORTCOUNT_A01` (Sport England "count" definition;
  0 = Inactive, 1 = Insufficiently Active, 2 = Active). Chosen because its national
  weighted active rate (**62.1%**) reproduces Sport England's published 2015–16
  headline (~61%), confirming the decoding and weighting are correct. The "fund"
  definition (54.6%) and pre-2016 definition (40.2%) were rejected as they do not
  match the published headline.
- **Weighting:** all percentages use the survey weight `wt_final`; no figure is a raw
  row count.
- **Missing data:** survey codes −90 to −99 recoded to missing before analysis.
- **Geography:** London identified via the ONS GSS code prefix `E09` embedded in the
  local-authority value labels, not borough names (avoids spelling mismatches).

### Data quality — audit results
The London analytic sample is **19,887 respondents across all 33 boroughs**. The audit
(`audit_wave`) confirms the data is clean and fit for analysis:

| Check | Result |
|---|---|
| Activity outcome missing | **0.0%** (complete — no respondent dropped from rates) |
| Survey weight missing | **0.0%**; all weights positive (min 0.018, max 37.0) |
| Ethnicity missing | 6.1% (item non-response) |
| Disability missing | 6.5% (item non-response) |
| Age / gender missing | 1.1% / 0.2% |
| Activity categories | Only the 3 valid labels — no leaked survey codes |
| Borough coverage | 33 of 33 London LAs present |

Missing values here reflect ordinary **item non-response** on optional/sensitive
questions (ethnicity, disability), not a processing error. They are reported, not
imputed — imputing would bias the very breakdowns under study. The activity outcome
and weight, which determine whether a figure is valid, are both complete.

### Headline results — Greater London, 2015–16
- **Overall active rate: 63.5%** (weighted).
- Unweighted activity counts: Active 12,687 / Insufficiently Active 2,476 / Inactive
  4,724. The raw active share (≈63.8%) is close to the weighted 63.5%, indicating the
  weighting does not dramatically reshape the London sample.
- London sits slightly **above** the national figure of 62.1% — plausibly reflecting
  London's younger age profile, but worth noting against the expectation that London
  has historically tracked around the England average.

### Activity by ethnicity (weighted % active; n = unweighted base)
| Ethnicity | % Active | n |
|---|---|---|
| Mixed | 73.4% | 522 |
| White Other | 68.9% | 2,855 |
| White British | 67.1% | 10,782 |
| Chinese | 61.1% | 319 |
| Black | 57.7% | 1,297 |
| Other ethnic group | 55.6% | 529 |
| South Asian | 54.7% | 2,365 |
| (Not recorded) | 59.3% | 1,218 |

A clear gradient: White and Mixed groups most active; South Asian, "Other", and Black
groups least active — consistent with established national patterns.

### Caveats
- **Small bases:** Chinese (n=319) and Mixed (n=522) rest on smaller samples, so those
  estimates are indicative. At borough level, City of London (n=267) is similarly small.
- **Missing ethnicity:** 1,218 respondents (~6%) have no recorded ethnicity; shown as a
  separate row rather than dropped, for transparency.
- These are **repeated cross-sections**, not the same individuals over time — suitable
  for tracking population trends, not individual change.

### Next steps
- Add the remaining waves (2016–17 … 2023–24) as new `WAVE_CONFIG` entries; the
  pipeline code is unchanged. Run `audit_wave` on each — the borough-count and
  weight-validity warnings will catch any renamed variable or wrong weight.
- **Harmonise** categories across waves (activity definition, ethnicity, age bands) via
  a mapping table so they are comparable year-on-year.
- Stack all waves into a single London panel for trend analysis and forecasting.